In [ ]:
%pip install requests beautifulsoup4 playwright nest_asyncio
%playwright install

   ---------------------------------------- 0.0/35.5 MB ? eta -:--:--
    --------------------------------------- 0.8/35.5 MB 11.4 MB/s eta 0:00:04
   --- ------------------------------------ 2.9/35.5 MB 8.9 MB/s eta 0:00:04
   ----- ---------------------------------- 4.7/35.5 MB 8.9 MB/s eta 0:00:04
   ------ --------------------------------- 6.0/35.5 MB 8.2 MB/s eta 0:00:04
   --------- ------------------------------ 8.1/35.5 MB 8.5 MB/s eta 0:00:04
   ------------ --------------------------- 11.0/35.5 MB 9.6 MB/s eta 0:00:03
   --------------- ------------------------ 14.2/35.5 MB 10.5 MB/s eta 0:00:03
   ------------------- -------------------- 17.3/35.5 MB 11.0 MB/s eta 0:00:02
   ------------------------ --------------- 21.8/35.5 MB 12.3 MB/s eta 0:00:02
   ---------------------------- ----------- 25.4/35.5 MB 12.9 MB/s eta 0:00:01
   ------------------------------- -------- 28.3/35.5 MB 13.0 MB/s eta 0:00:01
   ------------------------------------ --- 32.8/35.5 MB 13.8 MB/s eta 

In [ ]:
import os
import asyncio
import requests
from bs4 import BeautifulSoup
from pathlib import Path
from playwright.async_api import async_playwright

# === CONFIGURATION ===
toc_url = "https://practicalguidetoevil.wordpress.com/table-of-contents/"  # Replace with actual ToC link
output_dir = Path(os.getcwd())  # Save in same directory as script


# === SCRAPE TOC STRUCTURE ===
def get_books_and_links():
    response = requests.get(toc_url)
    soup = BeautifulSoup(response.text, 'html.parser')

    books = {}
    current_book = None

    for tag in soup.find_all(["h2", "ul"]):
        if tag.name == "h2" and "Book" in tag.get_text():
            current_book = tag.get_text(strip=True)
            books[current_book] = []
        elif tag.name == "ul" and current_book:
            for li in tag.find_all("li"):
                a = li.find("a")
                if a and a.get("href"):
                    chapter_title = a.get_text(strip=True)
                    chapter_link = a["href"]
                    books[current_book].append((chapter_title, chapter_link))
    return books


# === CLEAN FILE NAME ===
def sanitize_filename(title, index):
    import re
    title = re.sub(r'[^\w\s-]', '', title)  # Remove punctuation
    title = re.sub(r'\s+', '_', title.strip())  # Replace spaces
    return f"Chapter_{index:02d}_{title}.pdf"


# === PDF GENERATOR ===
async def save_books_as_pdfs(books):
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        context = await browser.new_context()
        
        for book_title, chapters in books.items():
            print(f"\n📘 Processing {book_title}...")

            # Make directory for book
            book_folder = output_dir / book_title.replace(" ", "_")
            book_folder.mkdir(exist_ok=True)

            for i, (chapter_title, url) in enumerate(chapters, start=1):
                print(f"  - Saving: {chapter_title} ...")

                try:
                    page = await context.new_page()
                    await page.goto(url, wait_until="networkidle")

                    file_name = sanitize_filename(chapter_title, i)
                    file_path = book_folder / file_name

                    await page.pdf(path=str(file_path), format="A4")
                    await page.close()
                except Exception as e:
                    print(f"    ⚠️ Failed to save {chapter_title}: {e}")

        await browser.close()


# === MAIN ===
if __name__ == "__main__":
    books = get_books_and_links()
    asyncio.run(save_books_as_pdfs(books))


📘 Book 1
  - Prologue: https://practicalguidetoevil.wordpress.com/2015/03/25/prologue/
  - Chapter 1: Knife: https://practicalguidetoevil.wordpress.com/2015/04/01/chapter-1-knife/
  - Chapter 2: Invitation: https://practicalguidetoevil.wordpress.com/2015/04/08/chapter-2-invitation/
  - Chapter 3: Party: https://practicalguidetoevil.wordpress.com/2015/04/16/chapter-3-squire/
  - Chapter 4: Name: https://practicalguidetoevil.wordpress.com/2015/04/22/chapter-4-name/
  - Chapter 5: Role: https://practicalguidetoevil.wordpress.com/2015/04/30/chapter-5-role/
  - Chapter 6: Aspect: https://practicalguidetoevil.wordpress.com/2015/05/06/chapter-6-aspect/
  - Chapter 7: Sword: https://practicalguidetoevil.wordpress.com/2015/05/13/chapter-seven-sword/
  - Chapter 8: Introduction: https://practicalguidetoevil.wordpress.com/2015/05/20/chapter-8-introduction/
  - Chapter 9: Claimant: https://practicalguidetoevil.wordpress.com/2015/05/27/chapter-9-claimant/
  - Chapter 10: Menace: https://practicalg